## Model Evaluation: Walk-Forward Cross Validation

Before comparing SARIMA forecasts against live World Cup data, the modeel's accuracy will be validated using walk-forward cross-validation. This ensures the baseline forecasts are reliable enough to meaningfully measure World Cup deviations.

**Methodology:**
- Split each city's time series into 4 folds
- Each fold trains on all data before the test window (no data leakage)
- Test window is 6 months per fold
- Metrics: RMSE and MAPE averaged across all folds and cities

In [11]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import TimeSeriesSplit 
from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.statespace.sarimax import SARIMAX
from pmdarima import auto_arima


In [12]:
# Reading in data
df = pd.read_csv('../data/processed/worldcup_city_seasonal_spending.csv', index_col='Date')
print(df.head())

            spend_all  spend_acf     cityname stateabbrev  city_pop2019  \
Date                                                                      
2022-01-31    -0.1300    -0.1580  Los Angeles          CA      10039107   
2022-01-31     0.1810     0.0257      Chicago          IL       5150233   
2022-01-31    -0.0201    -0.0351       Dallas          TX       2635516   
2022-01-31     0.0388     0.0112       Austin          TX       1273954   
2022-01-31    -0.0339    -0.0717    Charlotte          NC       1110356   

            season host_status  
Date                            
2022-01-31  Winter        Host  
2022-01-31  Winter    Non-Host  
2022-01-31  Winter        Host  
2022-01-31  Winter    Non-Host  
2022-01-31  Winter    Non-Host  


In [16]:
# Testing on Atlanta data to ensure pipeline works end-to-end and for debugging purposes

atl_df = df[df['cityname'] == 'Atlanta']
atl_spending = atl_df.spend_acf
print(atl_spending.head())
print(atl_spending.shape)

# Splitting data into 3 folds

time_split = TimeSeriesSplit(n_splits=2, test_size=12)  # 2 splits with a test size of 12 months each

# Loop through the splits and train/test the model
for train_idx, test_idx in time_split.split(atl_spending):
    atl_train, atl_test = atl_spending.iloc[train_idx], atl_spending.iloc[test_idx]
    print(f'Train shape: {atl_train.shape}, Test shape: {atl_test.shape}')

    # Fitting SARIMAX model
    auto_model = auto_arima(atl_train, seasonal=True, m=12, D=1, max_D=1, error_action='ignore', suppress_warnings=True)
    sarima_model = SARIMAX(atl_train, order=auto_model.order, seasonal_order=auto_model.seasonal_order)
    model_fit = sarima_model.fit(disp=False) 
    
    # Generate forecasts from training data
    forecast = model_fit.get_forecast(steps=len(atl_test))
    forecast_mean = forecast.predicted_mean
    forecast_mean.name = "Predicted Forecast"
    print(forecast_mean)    
    
    


Date
2022-01-31    0.0437
2022-02-28    0.1190
2022-03-31    0.0663
2022-04-30    0.1240
2022-05-31    0.0842
Name: spend_acf, dtype: float64
(53,)
Train shape: (29,), Test shape: (12,)
2024-06-30    0.058421
2024-07-31    0.043386
2024-08-31    0.012137
2024-09-30    0.088675
2024-10-31    0.132886
2024-11-30    0.063749
2024-12-31    0.111022
2025-01-31    0.074538
2025-02-28    0.130527
2025-03-31    0.078307
2025-04-30    0.105188
2025-05-31    0.111658
Freq: ME, Name: Predicted Forecast, dtype: float64
Train shape: (41,), Test shape: (12,)
2025-06-30    0.112666
2025-07-31    0.067701
2025-08-31    0.080203
2025-09-30    0.115444
2025-10-31    0.169025
2025-11-30    0.165844
2025-12-31    0.171453
2026-01-31    0.096265
2026-02-28    0.221998
2026-03-31    0.181637
2026-04-30    0.172406
2026-05-31    0.192259
Freq: ME, Name: Predicted Forecast, dtype: float64
